In [1]:
import pickle
import numpy as np
import pandas as pd
from enums import Frequency # import necessary for loading pickle file
from sklearn.model_selection import TimeSeriesSplit
from model_wrappers import wrapper_abstract, ets_wrapper
from model_wrappers.ets_wrapper import ExponentialSmoothingWrapper
from model_wrappers.naive_wrapper import NaiveForecastWrapper
from model_wrappers.autoets_wrapper import AutoETSWrapper

In [2]:
# load categorized timeseries data
with open('categorized_timeseries_dict.pkl', 'rb') as f:
    ts_all = pickle.load(f)

In [3]:
def get_num_of_splits(ts_length:int, ts_frequency:Frequency) -> int:
    # TODO: implement sensible split number logic based on timeseries length and sampling frequency
    return 5


In [5]:
def cross_validate_models(models:list, ts_dict:dict, use_boxcox:bool = True):
    for ts_name in ts_dict.keys():
        ts = ts_dict[ts_name]
        df = pd.DataFrame(ts['original'])
        df_boxcox = pd.DataFrame(ts['boxcox'])
        frequency = ts['frequency']
        values = df_boxcox.values.flatten() if use_boxcox else df.values.flatten()

        n_splits = get_num_of_splits(df.size, frequency)
        tscv = frequency.get_tscv(n_splits)
        period = frequency.get_period()

        model_squared_errors = [[] for _ in models]
        for train, test in tscv.split(values):
            model_index = 0
            for model in models:
                model.fit(data=values[train], seasonal_periods=period)
                forecast = model.forecast(steps=frequency.get_forecasting_horizon())
                target = values[test]
                error = np.sum((target - forecast)**2)
                model_squared_errors[model_index].append(error)
                model_index += 1
        mse = [np.mean(squared_error) for squared_error in model_squared_errors]
        print(np.argmin(mse))

ets = [
    ExponentialSmoothingWrapper(
        seasonal="mul",
        damped_trend=False,
        trend="add",
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal="mul",
        damped_trend=True,
        trend="add",
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal="add",
        trend="add",
        damped_trend=False,
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal="add",
        damped_trend=True,
        trend="add",
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=True,
        trend="add",
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=False,
        trend="add",
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal=None,
        damped_trend=False,
        trend=None,
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal="mul",
        damped_trend=False,
        trend=None,
        initialization_method="heuristic",),
    ExponentialSmoothingWrapper(
        seasonal="add",
        damped_trend=False,
        trend=None,
        initialization_method="heuristic",),
    NaiveForecastWrapper(
        is_seasonal=True
    ),
    NaiveForecastWrapper(
        is_seasonal=False
    )
]
#autoets_test = AutoETSWrapper(auto=True, allow_multiplicative_trend=True, n_jobs=-1)
cross_validate_models(ets, ts_all)

2
10
8
7
1
7
3
10
10
8
9


/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/exponential_smoothing/ets.py:1439: RuntimeWarning: invalid value encountered in divide
  self.standardized_forecasts_error = (


2
0
8
4
1
3
4
7
5
3
5
4
7
3
1
7


/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/

0


/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


4
4
10
2
8
0


/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/paul/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


9
0


KeyboardInterrupt: 